Initial NGSolve Practice Problem  

Given $u(x,y)=1-3x^2+2x^3$  

we set the boundary to Dirichlet =1 on the left side
and leave the others Neumann =0 default.  

Then we use $a(\nabla{u},\nabla{\phi})=(f,\phi)$ to solve for $u$.  

We set the RHS using the function then solve for our $u$.  

<span style="color:red">Current Concern: Direct Solve and CG Solve visually match, but don't match original function  
Consider error calculations in addition to visual inspection.</span>


In [36]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import ipywidgets as widgets  # might use to make plots near each other for visual comparison
import matplotlib.pyplot as plt
import numpy as np

In [37]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [38]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and default=Neumann on the other 3 sides
fes = H1(mesh, order=1, dirichlet="left")
fes.ndof

# Set function that will be used for boundary and RHS
g = 1 - 3 * x * x + 2 * x * x * x
Draw(g, mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [39]:
# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(g, BND) # this sets the Dirichlet boundary component to match the initial function (which is constant there), might want to modify later
Draw(gfu) 

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [40]:
# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# Bilinear form
a = BilinearForm(fes)
a += grad(u)*grad(phi)*dx # laplacian 
a.Assemble()

print(a.mat)

Row 0:   0: 1   4: -0.5   255: -0.5
Row 1:   1: 0.850708   66: -0.301993   67: -0.301217   318: -0.247498
Row 2:   2: 1   129: -0.5   130: -0.5
Row 3:   3: 0.850192   192: -0.300817   193: -0.300058   443: -0.249317
Row 4:   0: -0.5   4: 1.89558   5: -0.522363   255: -0.161358   256: -0.711859
Row 5:   4: -0.522363   5: 1.87514   6: -0.335775   256: -0.204528   257: -0.81248
Row 6:   5: -0.335775   6: 1.74323   7: -0.278355   257: -0.473294   258: -0.655805
Row 7:   6: -0.278355   7: 1.73308   8: -0.264766   258: -0.589395   259: -0.600561
Row 8:   7: -0.264766   8: 1.73389   9: -0.265657   259: -0.622531   260: -0.580939
Row 9:   8: -0.265657   9: 1.73373   10: -0.268822   260: -0.622295   261: -0.576953
Row 10:   9: -0.268822   10: 1.7333   11: -0.272926   261: -0.617083   262: -0.574466
Row 11:   10: -0.272926   11: 1.73287   12: -0.277969   262: -0.610804   263: -0.571173
Row 12:   11: -0.277969   12: 1.73249   13: -0.283054   263: -0.602387   264: -0.569077
Row 13:   12: -0.283054

In [41]:
# Right hand side
f = LinearForm(1 * phi * dx).Assemble()  # got this version from some example problem
print(f.vec)

 4.06901e-05
 6.52346e-05
 4.06901e-05
 6.51396e-05
 0.000144509
 0.000137263
 0.000111182
 0.000103239
 0.000101474
 0.000101734
 0.000102395
 0.000103216
 0.000104103
 0.000104952
 0.000105585
 0.000105988
 0.000106177
 0.00010623
 0.00010615
 0.000106046
 0.000105953
 0.000105862
 0.000105803
 0.000105754
 0.000105732
 0.00010572
 0.000105713
 0.000105711
 0.000105711
 0.000105712
 0.000105712
 0.000105712
 0.000105713
 0.000105713
 0.000105713
 0.000105713
 0.000105713
 0.000105713
 0.000105713
 0.000105714
 0.000105714
 0.000105714
 0.000105714
 0.000105714
 0.000105713
 0.000105712
 0.00010571
 0.000105706
 0.000105702
 0.000105697
 0.000105696
 0.000105692
 0.000105692
 0.000105681
 0.000105658
 0.00010561
 0.000105473
 0.0001052
 0.000104643
 0.000103702
 0.000102363
 0.000100802
 9.97944e-05
 9.96813e-05
 9.77403e-05
 9.13125e-05
 9.18285e-05
 9.18166e-05
 9.22882e-05
 0.000100866
 0.000104654
 0.000104927
 0.000104417
 0.00010376
 0.000103188
 0.000102719
 0.000102569
 0.0001

In [42]:
# Solve by using u=A^-1 (f-Au)
r = f.vec - a.mat * gfu.vec
gfu.vec.data += a.mat.Inverse(freedofs=fes.FreeDofs()) * r
Draw(gfu)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [43]:
# Solve the PDE using CG
gfu1 = GridFunction(fes)
gfu1.Set(g, BND)
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu1, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu1)

CG iteration 1, residual = 5.027160444849861     
CG iteration 2, residual = 2.5216856797764957     
CG iteration 3, residual = 1.702700658661104     
CG iteration 4, residual = 1.2991521968396278     
CG iteration 5, residual = 1.084892651788858     
CG iteration 6, residual = 0.9187121646291874     
CG iteration 7, residual = 0.775511010798235     
CG iteration 8, residual = 0.6776741555814725     
CG iteration 9, residual = 0.6089431566604193     
CG iteration 10, residual = 0.5544312337284597     
CG iteration 11, residual = 0.50563496852788     
CG iteration 12, residual = 0.46260097221018287     
CG iteration 13, residual = 0.4291519956342574     
CG iteration 14, residual = 0.40437223407282     
CG iteration 15, residual = 0.37938196507606614     
CG iteration 16, residual = 0.3531465618085168     
CG iteration 17, residual = 0.3335144273391585     
CG iteration 18, residual = 0.3174022370499568     
CG iteration 19, residual = 0.30453405181889287     
CG iteration 20, residual 

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene